# 🎵 Spotify Recommendation System (End-to-End)
Notebook ini berisi tahapan lengkap pembuatan sistem rekomendasi musik berbasis *Content-Based Filtering*, dari proses Load Data hingga Modeling, dibuat serapi mungkin.

## 1. Import Library & Load Data
Mengimpor library yang dibutuhkan dan mengunduh dataset dari Kaggle.

In [1]:
# Import library dasar
import os
import pandas as pd
import numpy as np

# Import library untuk Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

# Import kagglehub untuk download data
import kagglehub

# Download dataset
path = kagglehub.dataset_download("maharshipandya/-spotify-tracks-dataset")
dataset_path = os.path.join(path, "dataset.csv")

dataset = pd.read_csv(dataset_path)

dataset.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## 2. Data Cleaning
Membersihkan data dari kolom yang tidak penting, data kosong (*missing values*), dan duplikat lagu agar hasil rekomendasi lebih akurat.

In [2]:
# 1. Menghapus kolom 'Unnamed: 0' karena hanya berupa index tak bernilai
if 'Unnamed: 0' in dataset.columns:
    dataset = dataset.drop(['Unnamed: 0'], axis=1)

# 2. Menghapus baris yang memiliki nilai kosong (NaN)
dataset = dataset.dropna()

# 3. Menghapus lagu duplikat berdasarkan judul dan artis
# kita urutin berdasarkan poluper tertinggi agar versi terpopuler ttp dipertahankan
dataset = dataset.sort_values('popularity', ascending=False)
dataset = dataset.drop_duplicates(subset=['track_name', 'artists'], keep='first')

# 4. Reset index setelah penghapusan baris
dataset = dataset.reset_index(drop=True)

print(f"Data bersih siap digunakan! Jumlah lagu: {dataset.shape[0]}")

Data bersih siap digunakan! Jumlah lagu: 81343


## 3. Feature Engineering
Membuat sistem rekomendasi berdasarkan *Genre* dan *Artis*. Oleh karena itu, kita perlu menggabungkan kedua teks ini menjadi satu fitur (kolom `tags`).

In [3]:
# Menggabungkan genre dan artists menjadi satu teks utuh
dataset['tags'] = dataset['track_genre'] + " " + dataset['artists']

# Menampilkan hasil gabungan
dataset[['track_name', 'artists', 'track_genre', 'tags']].head()

,track_name,artists,track_genre,tags
0,Unholy (feat. Kim Petras),Sam Smith;Kim Petras,dance,dance Sam Smith;Kim Petras
1,"Quevedo: Bzrp Music Sessions, Vol. 52",Bizarrap;Quevedo,hip-hop,hip-hop Bizarrap;Quevedo
2,La Bachata,Manuel Turizo,reggae,reggae Manuel Turizo
3,I'm Good (Blue),David Guetta;Bebe Rexha,edm,edm David Guetta;Bebe Rexha
4,Tití Me Preguntó,Bad Bunny,latino,latino Bad Bunny


## 4. Modeling (TF-IDF & Cosine Similarity)
Model komputer tidak bisa membaca teks. Kita gunakan `TfidfVectorizer` untuk mengubah teks (tags) menjadi matriks angka. 
Lalu kita buat fungsi yang menggunakan `linear_kernel` (Cosine Similarity) untuk menghitung kemiripan lagu yang dicari dengan semua lagu di dataset.

In [4]:
# 1. Inisialisasi TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')

# 2. Mengubah teks 'tags' menjadi matriks vektor
tfidf_matrix = tfidf.fit_transform(dataset['tags'])
print(f"Bentuk matriks TF-IDF: {tfidf_matrix.shape}")

# 3. Membuat pemetaan judul lagu ke index untuk pencarian cepat
indices = pd.Series(dataset.index, index=dataset['track_name']).drop_duplicates()

# 4. Fungsi Sistem Rekomendasi
def get_recommendations(title, tfidf_matrix=tfidf_matrix, df=dataset, indices=indices):
    # Cek ketersediaan lagu
    if title not in indices:
        return "Lagu tidak ditemukan. Pastikan huruf besar/kecilnya sama persis."
        
    # Ambil index lagu
    idx = indices[title]
    if type(idx) == pd.Series:
        idx = idx.iloc[0] # ini buat ambil data yang pertama kalo misalnya ada 2 yang sama 
        
    # Hitung Cosine Similarity untuk lagu ini vs seluruh lagu lain
    sim_scores = linear_kernel(tfidf_matrix[idx], tfidf_matrix).flatten()
    
    # Ambil 10 urutan index dengan skor tertinggi (mengabaikan index 0)
    top_indices = sim_scores.argsort()[::-1][1:11]
    
    # Tampilkan hasilnya
    return df[['track_name', 'artists', 'track_genre']].iloc[top_indices]

Bentuk matriks TF-IDF: (81343, 29022)


## 5. Testing Sistem Rekomendasi


In [5]:
judul_lagu = 'Unholy (feat. Kim Petras)'

print(f"\n🎵 Rekomendasi lagu mirip '{judul_lagu}':")
get_recommendations(judul_lagu).head()


🎵 Rekomendasi lagu mirip 'Unholy (feat. Kim Petras)':


,track_name,artists,track_genre
74525,There Will Be Blood,Kim Petras,dance
75506,Massacre,Kim Petras,dance
75600,Close Your Eyes,Kim Petras,dance
76379,Tell Me It's A Nightmare,Kim Petras,dance
75531,Diamonds,Sam Smith,dance
